# Transformer baseline calculation

The project brief specifies 230 V RMS, 50 Hz, 100 VA, unity power factor, a 24 V RMS secondary target, 600:63 turns and an effective core area of 12 cm². This notebook evaluates a provisional E–I geometry and two unity-power-factor load conventions. The steel inputs come from the SIJ datasheet and catalogue. Wire sizes, dimensions, clearances, winding temperature and design limits are scenario choices. See [baseline inputs](../data/baseline_parameters.csv) and [development notes](../report/04_model_development.md).

In [1]:
from csv import DictReader
from math import ceil, floor, isclose
from pathlib import Path
import sys

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.constraints import DesignLimits, check_design
from src.core_geometry import EICore, WindowClearances
from src.material_models import Copper, HysteresisScenario, LaminatedSteel
from src.transformer_model import (
    ElectricalSpec, Winding, evaluate_design, ideal_secondary_voltage,
    nearest_secondary_turns,
)

with (root / "data" / "baseline_parameters.csv").open(encoding="utf-8", newline="") as handle:
    inputs = list(DictReader(handle))
parameters = {row["parameter"]: float(row["value"]) for row in inputs}
assert len(parameters) == len(inputs)
p = parameters


def turns(parameter):
    value = p[parameter]
    if not value.is_integer():
        raise ValueError(f"{parameter} must be an integer")
    return int(value)


assert p["load_power_factor"] == 1, "The resistive circuit assumes unity power factor"
spec = ElectricalSpec(p["primary_rms_v"], p["secondary_target_rms_v"],
                      p["frequency_hz"], p["rated_va"])
depth = p["effective_area_m2"] / (p["stacking_factor"] * p["centre_width_m"])
core = EICore(p["centre_width_m"], depth, p["window_width_m"], p["window_height_m"],
              p["stacking_factor"])
clearances = WindowClearances(p["former_wall_m"], p["outer_clearance_m"],
                              p["end_margin_m"], p["winding_barrier_m"])
primary, secondary = (
    Winding(turns(f"{name}_turns"), p[f"{name}_bare_diameter_m"],
            p[f"{name}_insulated_diameter_m"], p[f"{name}_lead_length_m"],
            p["winding_temperature_c"])
    for name in ("primary", "secondary")
)
copper = Copper(p["copper_resistivity_20_ohm_m"], p["copper_temp_coefficient_per_k"])
steel = LaminatedSteel(p["sheet_thickness_m"], p["steel_resistivity_ohm_m"],
                       p["steel_density_kg_m3"])
limits = DesignLimits(p["flux_limit_t"], p["maximum_supply_rms_v"], p["minimum_frequency_hz"],
                      p["maximum_fill_fraction"], p["ideal_voltage_tolerance_fraction"],
                      p["loaded_voltage_tolerance_fraction"], p["maximum_current_density_a_m2"])
print(f"Loaded {len(inputs)} labelled inputs from the baseline register.")

Loaded 44 labelled inputs from the baseline register.


## Integer turns, geometry and electrical operating points

The specified integer winding pair remains the baseline. The neighbouring integers demonstrate the ideal turns-ratio error. At load, the circuit solves both the specified rated secondary current and a separate exact 100 VA operating point. The winding loss comparison that follows uses the rated secondary current. The terminal voltage is a model prediction under the resistive assumptions.

In [2]:
ideal_turns = primary.turns * spec.secondary_target_rms_v / spec.primary_rms_v
nearest = nearest_secondary_turns(spec.primary_rms_v, spec.secondary_target_rms_v, primary.turns)
print(f"Ideal secondary turns: {ideal_turns:.4f}. Nearest positive integer: {nearest}.")
for candidate in sorted({max(1, floor(ideal_turns)), ceil(ideal_turns)}):
    voltage = ideal_secondary_voltage(spec.primary_rms_v, primary.turns, candidate)
    print(f"  {candidate} turns: {voltage:.3f} V ideal RMS ({voltage / spec.secondary_target_rms_v - 1:+.3%})")
assert nearest == secondary.turns

args = (spec, core, clearances, primary, secondary, copper, steel)
baseline = evaluate_design(*args)
exact_va = evaluate_design(*args, fixed_va=True)
assert isclose(core.effective_area_m2, p["effective_area_m2"])
assert isclose(exact_va.load.delivered_va, spec.rated_va)
print(f"Effective area: {core.effective_area_m2 * 1e4:.3f} cm²; gross stack depth: {depth * 1e3:.3f} mm")
print(f"One-loop magnetic path: {core.magnetic_path_m * 1e3:.3f} mm")
print(f"Steel volume: {core.steel_volume_m3 * 1e6:.3f} cm³; mass: {baseline.core_mass_kg:.4f} kg")
print(f"Mean primary/secondary turn length: {baseline.primary_layout.mean_turn_length_m * 1e3:.3f}"
      f" / {baseline.secondary_layout.mean_turn_length_m * 1e3:.3f} mm")
print(f"Ideal rating-based current references: {spec.primary_reference_a:.5f}"
      f" / {spec.secondary_reference_a:.5f} A RMS")
print(f"No-drop nominal flux density: {baseline.no_drop_peak_t:.5f} T peak")
for label, result in (("Rated secondary current", baseline), ("Exact 100 VA", exact_va)):
    load = result.load
    print(f"{label}: {load.secondary_terminal_rms_v:.4f} V RMS, {load.delivered_va:.4f} VA;"
          f" currents {load.primary_rms_a:.5f} / {load.secondary_rms_a:.5f} A RMS;"
          f" loaded flux density {result.loaded_peak_t:.5f} T peak")

Ideal secondary turns: 62.6087. Nearest positive integer: 63.
  62 turns: 23.767 V ideal RMS (-0.972%)
  63 turns: 24.150 V ideal RMS (+0.625%)
Effective area: 12.000 cm²; gross stack depth: 36.090 mm
One-loop magnetic path: 223.500 mm
Steel volume: 247.200 cm³; mass: 1.9010 kg
Mean primary/secondary turn length: 163.860 / 196.333 mm
Ideal rating-based current references: 0.43478 / 4.16667 A RMS
No-drop nominal flux density: 1.43801 T peak
Rated secondary current: 22.6584 V RMS, 94.4101 VA; currents 0.43750 / 4.16667 A RMS; loaded flux density 1.40420 T peak
Exact 100 VA: 22.5635 V RMS, 100.0000 VA; currents 0.46535 / 4.43194 A RMS; loaded flux density 1.40205 T peak


## Copper and classical eddy current losses

The steel loss is computed from loaded peak flux density. The nominal no-drop estimate above is used as a conservative reference for flux screening. The classical model predicts loss in an ideal insulated sheet with approximately uniform sinusoidal flux. The independent SIJ materials comparison uses approximate linear interpolation between two published typical anchors and assumes \(B\approx J\) over that interval. It is a sheet-level check with an unresolved remainder.

In [3]:
print(f"Primary and secondary resistance: {baseline.primary_ohm:.5f} / {baseline.secondary_ohm:.5f} Ω")
print(f"Copper loss, primary and secondary: {baseline.primary_copper_w:.4f}"
      f" / {baseline.secondary_copper_w:.4f} W")
print(f"Copper subtotal: {baseline.copper_w:.4f} W")
print(f"Classical eddy current loss: {baseline.classical_w:.4f} W")
print(f"Subtotal before hysteresis: {baseline.copper_w + baseline.classical_w:.4f} W")

anchor_f = p["anchor_frequency_hz"]
anchor_low = p["anchor_low_polarisation_t"]
anchor_high = p["anchor_high_polarisation_t"]
anchor_low_loss = p["anchor_low_total_loss_w_kg"]
anchor_high_loss = p["anchor_high_total_loss_w_kg"]
assert anchor_low < anchor_high
assert isclose(spec.frequency_hz, anchor_f), "The manufacturer comparison applies at its test frequency"
assert anchor_low <= baseline.loaded_peak_t <= anchor_high, "Loaded induction outside comparison anchors"
weight = (baseline.loaded_peak_t - anchor_low) / (anchor_high - anchor_low)
interpolated_total = anchor_low_loss + weight * (anchor_high_loss - anchor_low_loss)
classical_specific = baseline.classical_w / baseline.core_mass_kg
print(f"SIJ anchor interpolation (sheet total only): {interpolated_total:.4f} W/kg")
print(f"Unresolved remainder after classical estimate: {interpolated_total - classical_specific:.4f} W/kg")
assert isclose(baseline.classical_w,
               steel.classical_loss_w_m3(spec.frequency_hz, baseline.loaded_peak_t) * core.steel_volume_m3)

Primary and secondary resistance: 12.35845 / 0.22173 Ω
Copper loss, primary and secondary: 2.3655 / 3.8494 W
Copper subtotal: 6.2149 W
Classical eddy current loss: 0.5819 W
Subtotal before hysteresis: 6.7968 W
SIJ anchor interpolation (sheet total only): 2.3411 W/kg
Unresolved remainder after classical estimate: 2.0350 W/kg


## Illustrative hysteresis sensitivity

At the 1.5 T comparison anchor, subtract the calculated classical term from the typical total. Allocate 25%, 50% or 75% of that unresolved power to hysteresis as a deliberately illustrative range. Divide by the reference frequency to get quasi-static energy per cycle, then scale energy as \((B/1.5\,\mathrm{T})^2\) over 1.0–1.5 T. This allocation and exponent are scenario inputs. They are not fitted coefficients, physical bounds or a manufacturer-backed decomposition. The SIJ total interpolation is independent of the separated objective and is never added to it.

In [4]:
reference_remainder = anchor_high_loss - steel.classical_loss_w_kg(anchor_f, anchor_high)
assert reference_remainder > 0
assert baseline.separated_total_w is None
for label in ("low", "mid", "high"):
    fraction = p[f"hysteresis_{label}_fraction"]
    assert 0 <= fraction <= 1
    scenario = HysteresisScenario(
        f"Illustrative {label}", fraction * reference_remainder / anchor_f, anchor_high,
        p["hysteresis_exponent"], anchor_low, anchor_high,
    )
    result = evaluate_design(*args, scenario)
    print(f"{scenario.label:17s} ({fraction:>4.0%} at anchor): "
          f"{result.hysteresis_w:.4f} W hysteresis; {result.separated_total_w:.4f} W three-loss sum")

Illustrative low  ( 25% at anchor): 0.9540 W hysteresis; 7.7508 W three-loss sum
Illustrative mid  ( 50% at anchor): 1.9081 W hysteresis; 8.7048 W three-loss sum
Illustrative high ( 75% at anchor): 2.8621 W hysteresis; 9.6589 W three-loss sum


## Feasibility screens

The limits below are assumptions for this calculation. Each winding uses insulated diameter for layer geometry and bare copper area for the fill and current-density tests. The flux envelope is a provisional 241.5 V RMS at 50 Hz. No-load excitation, leakage, steel-joint effects and temperature feedback are outside this model. Screen results apply only to these inputs.

In [5]:
def for_display(value, unit):
    if unit == "fraction":
        return f"{value:.3%}"
    if unit == "A/m²":
        return f"{value / 1e6:.3f} A/mm²"
    if unit == "m":
        return f"{value * 1e3:.3f} mm"
    return f"{value:.4f} {unit}"


print(f"Primary/secondary layers: {baseline.primary_layout.layers} / {baseline.secondary_layout.layers}")
print(f"Usable one-window area: {baseline.usable_window_m2 * 1e6:.2f} mm²")
checks = check_design(baseline, limits)
for check in checks:
    verdict = "PASS" if check.passed else "FAIL"
    print(f"{verdict} | {check.name}: {for_display(check.value, check.unit)} "
          f"(limit {for_display(check.limit, check.unit)})")
print(f"All baseline screens passed: {all(check.passed for check in checks)}")

Primary/secondary layers: 7 / 2
Usable one-window area: 704.00 mm²
FAIL | Peak flux at screening envelope: 1.5099 T (limit 1.5000 T)
PASS | Bare-copper window fill: 23.676% (limit 35.000%)
PASS | Insulated radial build: 7.100 mm (limit 16.000 mm)
PASS | Ideal secondary voltage error: 0.625% (limit 1.000%)
FAIL | Loaded secondary voltage error: 5.590% (limit 5.000%)
PASS | Primary current density: 2.751 A/mm² (limit 4.000 A/mm²)
PASS | Secondary current density: 3.684 A/mm² (limit 4.000 A/mm²)
All baseline screens passed: False
